In [2]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()
project_root = cwd if (cwd / "src").is_dir() else cwd.parent

if not (project_root / "src").is_dir():
    raise FileNotFoundError(f"Không tìm thấy thư mục src từ: {cwd}")

project_root_str = str(project_root)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Project root: {project_root}")

Project root: /home/trieu/Intern/SwM_precomputed


In [3]:
import torch

from torch.utils.data import DataLoader

from src.utils.io import load_pickle
from src.models.sasrec import SASRec
from src.training.train import train_model
from src.training.loss import SASRecLoss
from src.data.train_dataset import TrainDataset
from src.data.evaluation_dataset import EvaluationDataset


In [4]:
train_samples = [
    {
        'history': [
            'N8129',
            'N1569',
            'N17686',
        ],
        'target': 'N13008',
    },
    {
        'history': [
            'N63302',
            'N10414',
            'N19347',
            'N31801'
        ],
        'target': 'N55689'
    },
    {
        'history': [
            'N21623',
            'N6233',
            'N14340',
            'N48031',
            'N62285'
        ],
        'target': 'N31739'
    },
    {
        'history': [
            'N31739',
            'N6072',
            'N63045',
            'N23979',
            'N35656',
        ],
        'target': 'N43353'
    },
    
]

In [5]:
padding_id = 0
max_sequence_length = 5
evaluation_samples = []
num_negatives = 5
mapping = load_pickle(project_root / "data/processed/mindsmall_v2/artifacts/news_vector_mapping.pkl")
batch_size = 2
num_blocks = 2
num_heads = 2
dropout = 0.1
embedding_dim = 384
lr = 0.001
k = 5

In [6]:
train_dataset = TrainDataset(samples=train_samples, max_sequence_length=max_sequence_length, padding_id=padding_id, mapping=mapping, vector_size=384)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [7]:
model = SASRec(
    max_sequence_length=max_sequence_length,
    num_blocks=num_blocks,
    num_heads=num_heads,
    dropout=dropout,
    embedding_dim=embedding_dim
)

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

In [9]:
val_samples = [
    {
        'history': ['N41241', 'N52026', 'N36360', 'N31550'],
        'target': 'N26125'
    },
    {
        'history': ['N59027'], 
        'target': 'N54803'
    },
    {
        'history': ['N39556','N459','N57300','N4643','N16545','N4501','N10059','N719','N51483','N56446','N32868','N19620','N61084','N10865','N16625','N19638','N31193','N25577','N12096','N48216','N20110','N18275','N28115','N25525','N29453','N16715','N24298','N18355','N4985','N6506','N14761','N1864','N8148','N46811'],
        'target': 'N5981'
    },
    {
        'history': ['N39556','N459','N57300','N4643','N16545','N4501','N10059','N719','N51483','N56446','N32868','N19620','N61084','N10865','N16625','N19638','N31193','N25577','N12096','N48216','N20110','N18275','N28115','N25525','N29453','N16715','N24298','N18355','N4985','N6506','N14761','N1864','N8148','N46811'],
        'target': 'N16120'
    },
]

In [10]:
val_dataset = EvaluationDataset(
    samples=val_samples,
    num_negatives=num_negatives,
    max_sequence_length=max_sequence_length,
    padding_id=padding_id,
    mapping=mapping,
    vector_size=384
)
val_dataloader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [11]:
criterion = SASRecLoss()

In [12]:
from src.utils.io import create_run_directory

experiment_name = "test"
run_dir = create_run_directory("outputs", experiment_name)


In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

model = model.to(device)

results = train_model(
    model=model,
    train_loader=train_loader,
    validation_loader=val_dataloader,
    criterion=SASRecLoss(),
    optimizer=optimizer,
    device=device,
    num_epochs=6,
    k=k,
    patience=2,
    min_delta=0.001,
    run_dir=run_dir,
    model_config=None,
    max_norm=5.0
)

Using device: cuda


/home/trieu/Intern/SwM_precomputed/src/data/train_dataset.py:75: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  "input_vectors": torch.tensor(


Epoch 1/6 - Loss: 1.4577 - HR@5: 1.0000 - NDCG@5: 0.4827
Saved new best model at epoch 1.
Epoch 2/6 - Loss: 0.4495 - HR@5: 1.0000 - NDCG@5: 0.5982
Saved new best model at epoch 2.
Epoch 3/6 - Loss: 0.3384 - HR@5: 1.0000 - NDCG@5: 0.6905
Saved new best model at epoch 3.
Epoch 4/6 - Loss: 0.3690 - HR@5: 1.0000 - NDCG@5: 0.7827
Saved new best model at epoch 4.
Epoch 5/6 - Loss: 0.4065 - HR@5: 1.0000 - NDCG@5: 0.6905
No improvement. Patience: 1/2
Epoch 6/6 - Loss: 0.2943 - HR@5: 1.0000 - NDCG@5: 0.5982
No improvement. Patience: 2/2
Early stopping at epoch 6.


In [15]:
test_samples = [
    {
        'history': ['N33998','N47765','N56742','N36511','N44796','N22141','N21895','N23718','N32852','N20201','N19615','N8745','N27766','N22816'],
        'target': 'N53572'
    },
    {
        'history': ['N44251','N27612','N36699','N40467','N26015','N61997','N57336','N59704','N41987','N59691','N46990','N37182','N30967','N18285','N11005','N306','N51821','N36530','N42620','N50','N55714','N11020'],
        'target': 'N46162'
    },
    {
        'history': ['N33276', 'N2186', 'N250', 'N34323', 'N15476', 'N36424'],
        'target': 'N62365'
    },
    {
        'history': ['N24073','N6974','N29911','N63019','N5391','N54496','N33683','N33454','N27612','N3567','N8082','N30389','N1603','N59465','N33276','N841','N19636','N4593','N12709','N23249','N11116','N43955','N43339','N55266','N41429','N37091','N8148','N15676'],
        'target': 'N56969'
    }
]

In [17]:
test_dataset = EvaluationDataset(
    samples=test_samples,
    num_negatives=num_negatives,
    max_sequence_length=max_sequence_length,
    padding_id=padding_id,
    mapping=mapping,
    vector_size=384
)
test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

In [20]:
from src.training.evaluate import evaluate_ranking


test_metrics = evaluate_ranking(
    model,
    test_dataloader,
    device,
    k
)
test_metrics

{'hit_rate@5': 0.75, 'ndcg@5': 0.4434264227747917}